# Módulo de Previsão de Séries Temporais Financeiras Usando Redes Neurais

## Setup das Bibliotecas

In [1]:
################################################################################################################################
########## Importando Bibliotecas Básicas ######################################################################################
################################################################################################################################
import sys
import gc
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import time
import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch
import optuna
import tempfile
import mlflow

In [3]:
from utilsforecast.losses import *
from utilsforecast.processing import *
from utilsforecast.evaluation import *
from utilsforecast.feature_engineering import *

In [4]:
from functools import partial
from functools import reduce

In [5]:
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MQLoss, DistributionLoss, HuberMQLoss, MAE, SMAPE, PMM, MSE, GMM, RMSE, HuberLoss
from neuralforecast.models import NBEATSx, NHITS, DeepAR, NLinear, LSTM, RNN, DLinear, BiTCN, TFT, PatchTST, iTransformer, TSMixer

In [6]:
from typing import Tuple,List, Dict
from datetime import timedelta

In [7]:
import os

os.environ['NIXTLA_ID_AS_COL'] = '1'
# don't reset the index on the output from predict if that's what you're currently doing

In [8]:
import lightning.pytorch as pl
from lightning.pytorch.loggers import CSVLogger

# Disable logging
trainer_no_log = pl.Trainer(logger=False)

# Use a specific logger
csv_logger = CSVLogger("logs", name="my_experiment")
trainer_custom_log = pl.Trainer(logger=csv_logger)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [9]:
# Detecta o melhor acelerador disponível: CUDA, ROCm, MPS ou CPU
def _detectar_acelerador():
    # CUDA (NVIDIA) e ROCm (AMD) usam a mesma API torch.cuda no PyTorch
    if torch.cuda.is_available():
        return 'gpu', [0]
    # MPS (Apple Silicon / Metal)
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return 'mps', 1
    # Fallback para CPU
    return 'cpu', 1

_accelerator, _devices = _detectar_acelerador()

# Argumentos do treinador para os modelos da NeuralForecast
TRAINER_KWARGS = {
    'accelerator': _accelerator,
    'devices': _devices,
}

## Declaração dos Modelos

In [ ]:
H = 15
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]

# ---------- catálogo de losses (FUNÇÃO DE PERDA DE TREINO) -------------------

LOSSES = {
    "normal":  DistributionLoss(distribution="Normal",
                                quantiles=QUANTILES, return_params=False),
   # "gamma":   DistributionLoss(distribution="Gamma",
    #                            quantiles=QUANTILES, return_params=False),
   # "weibull": DistributionLoss(distribution="Weibull",
    #                            quantiles=QUANTILES, return_params=False),
    "mq":      MQLoss(quantiles=QUANTILES),
    "huber":   HuberLoss(delta=1.0),
}

# Cada loss é distribuição ou ponto.
IS_DISTRIBUTION = {
    "normal":  True,
    "gamma":   True,
    "weibull": True,
    "mq":      False,
    "huber":   False,
}

# Gamma e Weibull exigem y > 0 — sem zeros, sem negativos.
REQUIRES_POSITIVE = {"gamma", "weibull"}


# ---------- helpers de config (espaço de busca por trial) -------------------

def _base_train(trial):
    return {
        "hist_exog_list":['rsi_14', 'macd_hist', 'bb_pct', 'atr_norm', 'obv_z63', 'vwap_rel21', 'mom_252_21', 'mom_126_21', 'mom_63_21', 'reversal_21', 'high_252w', 'rvol_63', 'rvol_21', 'skew_63', 'kurt_63', 'max_ret_21', 'min_ret_21', 'amihud_21', 'turnover_21', 'beta_mkt_rf', 'beta_vix', 'beta_brl_ret', 'rf_br', 'mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf_us', 'vix', 'brl_ret', 'dxy_ret'], 
        "futr_exog_list" : Y_df_train.drop(['y','unique_id','ds','rsi_14', 'macd_hist', 'bb_pct', 'atr_norm', 'obv_z63', 'vwap_rel21', 'mom_252_21', 'mom_126_21', 'mom_63_21', 'reversal_21', 'high_252w', 'rvol_63', 'rvol_21', 'skew_63', 'kurt_63', 'max_ret_21', 'min_ret_21', 'amihud_21', 'turnover_21', 'beta_mkt_rf', 'beta_vix', 'beta_brl_ret', 'rf_br', 'mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf_us', 'vix', 'brl_ret', 'dxy_ret'],axis=1).columns.values,
        "max_steps":     trial.suggest_categorical("max_steps", [5000, 10000, 20000]),
        "batch_size":    trial.suggest_categorical("batch_size", [32, 64, 128]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
        "scaler_type":   trial.suggest_categorical("scaler_type",
                                                   ["robust", "standard"]),  # "identity" diverge com exógenas de grande magnitude
        "random_seed":   trial.suggest_int("random_seed", 1, 10_000),
        "early_stop_patience_steps": 10,
        "val_check_steps":           50,
    }

def _input_size(trial, mults=(1, 2, 3, 5, 7)):
    return trial.suggest_categorical("input_size", [m * H for m in mults])


def nhits_cfg(trial):
    return {**_base_train(trial),
            "input_size":         _input_size(trial),
            "n_pool_kernel_size": trial.suggest_categorical(
                "n_pool_kernel_size",
                [[2,2,2], [4,4,4], [8,4,1], [16,8,1]]),
            "n_freq_downsample":  trial.suggest_categorical(
                "n_freq_downsample",
                [[168,24,1], [24,12,1], [1,1,1]]),
            "mlp_units":          trial.suggest_categorical(
                "mlp_units", [3*[[256,256]], 3*[[512,512]]]),
            "dropout_prob_theta": trial.suggest_float("dropout_prob_theta", 0.0, 0.3)}

def nbeatsx_cfg(trial):
    return {**_base_train(trial),
            "input_size":  _input_size(trial),
            "n_blocks":    trial.suggest_categorical(
                "n_blocks", [[1,1,1], [2,2,2], [3,3,3]]),
            "mlp_units":   trial.suggest_categorical(
                "mlp_units", [3*[[256,256]], 3*[[512,512]]]),
            "stack_types": ["identity", "trend", "seasonality"]}

def tft_cfg(trial):
    return {**_base_train(trial),
            "input_size":   _input_size(trial),
            "hidden_size":  trial.suggest_categorical("hidden_size", [16, 32, 64]),  # TFT escala com hidden_size x nº de exógenas -> mantenha pequeno p/ caber na VRAM
            "n_head":       trial.suggest_categorical("n_head", [2, 4, 8]),
            "dropout":      trial.suggest_float("dropout", 0.0, 0.3),
            "attn_dropout": trial.suggest_float("attn_dropout", 0.0, 0.3)}

def lstm_cfg(trial):
    return {**_base_train(trial),
            "input_size":          -1,
            "encoder_hidden_size": trial.suggest_categorical(
                "encoder_hidden_size", [64, 128, 256]),
            "encoder_n_layers":    trial.suggest_categorical(
                "encoder_n_layers", [1, 2, 3]),
            "context_size":        trial.suggest_categorical(
                "context_size", [5, 10, 20]),
            "decoder_hidden_size": trial.suggest_categorical(
                "decoder_hidden_size", [64, 128, 256])}

def rnn_cfg(trial):
    return {**_base_train(trial),
            "input_size":          -1,
            "encoder_hidden_size": trial.suggest_categorical(
                "encoder_hidden_size", [64, 128, 256]),
            "encoder_n_layers":    trial.suggest_categorical(
                "encoder_n_layers", [1, 2, 3]),
            "encoder_activation":  trial.suggest_categorical(
                "encoder_activation", ["tanh", "relu"]),
            "encoder_dropout":     trial.suggest_float("encoder_dropout", 0.0, 0.3),
            "context_size":        trial.suggest_categorical(
                "context_size", [5, 10, 20]),
            "decoder_hidden_size": trial.suggest_categorical(
                "decoder_hidden_size", [64, 128, 256])}

def patchtst_cfg(trial):
    return {**_base_train(trial),
            "input_size":     _input_size(trial),
            "hidden_size":    trial.suggest_categorical("hidden_size", [64, 128, 256]),
            "n_heads":        trial.suggest_categorical("n_heads", [4, 8, 16]),
            "patch_len":      trial.suggest_categorical("patch_len", [8, 16, 24]),
            "stride":         trial.suggest_categorical("stride", [4, 8, 16]),
            "encoder_layers": trial.suggest_categorical("encoder_layers", [2, 3, 4]),
            "dropout":        trial.suggest_float("dropout", 0.0, 0.3)}

def itransformer_cfg(trial):
    return {**_base_train(trial),
            "input_size":  _input_size(trial),
            "hidden_size": trial.suggest_categorical("hidden_size", [128, 256, 512]),
            "n_heads":     trial.suggest_categorical("n_heads", [4, 8]),
            "e_layers":    trial.suggest_categorical("e_layers", [2, 3, 4]),
            "d_ff":        trial.suggest_categorical("d_ff", [256, 512, 1024]),
            "dropout":     trial.suggest_float("dropout", 0.0, 0.3)}

def tsmixer_cfg(trial):
    return {**_base_train(trial),
            "input_size": _input_size(trial),
            "n_block":    trial.suggest_categorical("n_block", [2, 4, 6]),
            "ff_dim":     trial.suggest_categorical("ff_dim", [64, 128, 256]),
            "dropout":    trial.suggest_float("dropout", 0.0, 0.3)}

def deepar_cfg(trial):
    return {**_base_train(trial),
            "input_size":       _input_size(trial),
            "lstm_hidden_size": trial.suggest_categorical(
                "lstm_hidden_size", [64, 128, 256]),
            "lstm_n_layers":    trial.suggest_categorical(
                "lstm_n_layers", [1, 2, 3]),
            "lstm_dropout":     trial.suggest_float("lstm_dropout", 0.0, 0.3)}

def bitcn_cfg(trial):
    return {**_base_train(trial),
            "input_size":  _input_size(trial),
            "hidden_size": trial.suggest_categorical("hidden_size", [16, 32, 64, 128]),
            "dropout":     trial.suggest_float("dropout", 0.0, 0.5)}


# Catálogo: nome do modelo -> função de config (trial -> dict de hiperparâmetros)
MODEL_REGISTRY = {
    "NHITS":        nhits_cfg,
    "NBEATSx":      nbeatsx_cfg,
    "TFT":          tft_cfg,
    "LSTM":         lstm_cfg,
    "RNN":          rnn_cfg,
    "PatchTST":     patchtst_cfg,
    "iTransformer": itransformer_cfg,
    "TSMixer":      tsmixer_cfg,
    "DeepAR":       deepar_cfg,
    "BiTCN":        bitcn_cfg,
}

# Orçamento de busca (nº de trials do Optuna)
NUM_TRIALS = 50

In [11]:
# =============================================================================
# Infraestrutura de acompanhamento no MLflow
# =============================================================================
# Requisitos:  pip install mlflow plotly
def setup_mlflow(experiment_name, tracking_uri=None):
    """Configura o destino do tracking e o experimento ativo."""
    if tracking_uri:
        mlflow.set_tracking_uri(tracking_uri)

    # Fazemos o logging manualmente (um run por trial). Desligamos o autolog do
    # PyTorch Lightning para evitar runs duplicados e o warning de checkpoint.
    try:
        mlflow.autolog(disable=True)
    except Exception as e:
        print(f"[mlflow] nao foi possivel desativar o autolog: {e}")

    mlflow.set_experiment(experiment_name)
    print(f"[mlflow] tracking_uri = {mlflow.get_tracking_uri()}")
    print(f"[mlflow] experiment   = {experiment_name}")


def _log_optuna_plots(study):
    """Salva gráficos do Optuna (histórico + importância) como artefatos."""
    try:
        import optuna.visualization as vis
        with tempfile.TemporaryDirectory() as tmp:
            figs = {
                "optuna_history.html":     vis.plot_optimization_history,
                "optuna_importances.html": vis.plot_param_importances,
                "optuna_parallel.html":    vis.plot_parallel_coordinate,
            }
            for fname, fn in figs.items():
                try:
                    path = os.path.join(tmp, fname)
                    fn(study).write_html(path)
                    mlflow.log_artifact(path, artifact_path="optuna")
                except Exception as e:
                    print(f"[mlflow] gráfico {fname} ignorado: {e}")
    except Exception as e:
        print(f"[mlflow] visualizações do Optuna indisponíveis (instale plotly): {e}")

In [ ]:
# =============================================================================
# Sweep otimizando SCORE PROBABILÍSTICO (CRPS), com log ao vivo + gestão de VRAM
# =============================================================================
import gc
import numpy as np
from scipy.special import erf  # scipy é dependência do neuralforecast


BASE_MODELS = {
    "NHITS": NHITS, "NBEATSx": NBEATSx, "TFT": TFT, "LSTM": LSTM, "RNN": RNN,
    "PatchTST": PatchTST, "iTransformer": iTransformer,
    "TSMixer": TSMixer, "DeepAR": DeepAR, "BiTCN": BiTCN,
}

# Qual score probabilístico otimizar: "crps" (minimiza) ou "loglik" (maximiza)
PROB_SCORE = "crps"
_DIRECTION = {"crps": "minimize", "loglik": "maximize"}

# --------------------------- Controle de VRAM -------------------------------
PRECISION      = "bf16-mixed"   # mixed precision ~metade da VRAM; use "16-mixed" se a GPU não tiver bf16, ou "32-true" p/ desligar
MAX_BATCH_SIZE = 16             # teto de batch_size (limita VRAM)
MAX_INPUT_SIZE = 3 * H          # teto de input_size (janelas longas pesam muito)
RECURRENT_INPUT_SIZE = 3 * H    # janela FINITA p/ recorrentes (LSTM/RNN) que pedem input_size=-1 (sequência inteira -> OOM)

# Modelos recorrentes: bf16/16-mixed deixa o scale da DistributionLoss virar NaN.
# Rodamos esses em fp32 ("32-true") e o resto em PRECISION (bf16-mixed).
RECURRENT_MODELS   = {"LSTM", "RNN", "DeepAR"}
PRECISION_OVERRIDE = {m: "32-true" for m in RECURRENT_MODELS}

def _precision_for(model_name=None):
    return PRECISION_OVERRIDE.get(model_name, PRECISION)


def _free_gpu():
    """Libera memória de GPU acumulada entre trials."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _trainer_kwargs(model_name=None):
    """Kwargs do Lightning Trainer (repassados pelo modelo) p/ economizar memória."""
    acc, dev = _detectar_acelerador()
    return dict(
        accelerator=acc,
        devices=dev,
        precision=_precision_for(model_name),
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        gradient_clip_val=1.0,   # evita explosão de gradiente -> NaN
    )


def _is_oom(exc):
    return isinstance(exc, getattr(torch.cuda, "OutOfMemoryError", ())) or (
        isinstance(exc, RuntimeError) and "out of memory" in str(exc).lower()
    )


def _is_divergence(exc):
    """Divergência numérica: parâmetros viraram NaN/Inf durante o treino."""
    msg = str(exc).lower()
    return isinstance(exc, ValueError) and (
        "invalid values" in msg or "satisfy the constraint" in msg or "nan" in msg
    )


def _norm_cdf(z):
    return 0.5 * (1.0 + erf(z / np.sqrt(2.0)))

def _norm_pdf(z):
    return np.exp(-0.5 * z * z) / np.sqrt(2.0 * np.pi)


def _compute_metrics(y, yhat):
    """Métricas de ERRO (referência), sobre a previsão de ponto (média)."""
    err = y - yhat
    ae = np.abs(err)
    denom = np.abs(y) + np.abs(yhat) + 1e-8
    return {
        "mae":   float(np.mean(ae)),
        "rmse":  float(np.sqrt(np.mean(err ** 2))),
        "bias":  float(np.mean(err)),
        "smape": float(np.mean(2.0 * ae / denom)),
    }


def _find_param_col(df, alias, suffix):
    cands = [c for c in df.columns if c.startswith(alias) and c.endswith(suffix)]
    if not cands:
        raise KeyError(
            f"Coluna '{suffix}' não encontrada para '{alias}' "
            f"(precisa de return_params=True). Colunas: {list(df.columns)}")
    return cands[0]


def _make_dist_loss_with_params(loss_name):
    """Recria a loss de distribuição escolhida com return_params=True."""
    if not IS_DISTRIBUTION.get(loss_name, False):
        raise ValueError(
            f"Score probabilístico paramétrico requer loss de distribuição; "
            f"'{loss_name}' não é. Use LOSS_NAME='normal' (ou veja CRPS por quantis).")
    base = LOSSES[loss_name]
    distribution = getattr(base, "distribution", None) or "Normal"
    return DistributionLoss(distribution=distribution,
                            quantiles=QUANTILES, return_params=True), distribution


def _prob_scores_normal(y, loc, scale):
    """CRPS (forma fechada) e log-verossimilhança média sob a Normal prevista."""
    scale = np.clip(scale, 1e-6, None)
    z = (y - loc) / scale
    crps = scale * (z * (2.0 * _norm_cdf(z) - 1.0)
                    + 2.0 * _norm_pdf(z) - 1.0 / np.sqrt(np.pi))
    loglik = -0.5 * np.log(2 * np.pi) - np.log(scale) - 0.5 * z ** 2
    return {"crps": float(np.mean(crps)), "loglik": float(np.mean(loglik))}


def _crps_from_quantiles(y, q_preds, quantiles):
    """CRPS distribution-free = 2 * média do pinball loss sobre a grade de quantis.

    `q_preds`: array (n_obs, n_quantis); `quantiles`: lista de níveis em (0,1).
    Útil quando não há parâmetros (ex.: MQLoss).
    """
    y = y[:, None]
    q = np.asarray(quantiles)[None, :]
    e = y - q_preds
    pinball = np.maximum(q * e, (q - 1.0) * e)   # quantile/pinball loss
    return float(2.0 * np.mean(pinball))


def _cap_for_vram(cfg):
    """Limita os hiperparâmetros que mais consomem VRAM (sem quebrar o trial)."""
    if cfg.get("batch_size") and cfg["batch_size"] > MAX_BATCH_SIZE:
        cfg["batch_size"] = MAX_BATCH_SIZE
    isz = cfg.get("input_size")
    if isinstance(isz, int):
        if isz == -1:                       # recorrentes (LSTM/RNN): -1 = sequência inteira -> OOM
            cfg["input_size"] = RECURRENT_INPUT_SIZE
        elif isz > MAX_INPUT_SIZE:
            cfg["input_size"] = MAX_INPUT_SIZE
    return cfg


def run_sweep_prob(Y_df, model_name, loss_name, prob_score=PROB_SCORE,
                   h=H, num_trials=NUM_TRIALS, val_size=None, freq="D",
                   n_windows=1, experiment_name=None, tracking_uri=None):
    """Sweep que OTIMIZA um score probabilístico (CRPS/loglik), com log ao vivo.

    Robusto a OOM: libera a GPU a cada trial e, se faltar VRAM, marca o trial
    como podado (optuna.TrialPruned) em vez de derrubar o estudo inteiro.
    Retorna (study, run_id).
    """
    val_size = val_size if val_size is not None else h * 4
    experiment_name = experiment_name or f"forecast_global__{model_name}__{loss_name}__{prob_score}"
    setup_mlflow(experiment_name, tracking_uri)

    cls = BASE_MODELS[model_name]
    cfg_fn = MODEL_REGISTRY[model_name]
    loss_params, distribution = _make_dist_loss_with_params(loss_name)
    if distribution != "Normal":
        raise NotImplementedError(
            f"Forma fechada de CRPS/loglik implementada só para 'Normal' "
            f"(recebido '{distribution}'). Para outras, use _crps_from_quantiles.")

    df = Y_df
    if loss_name in REQUIRES_POSITIVE:
        n_before = len(df)
        df = df[df["y"] > 0].copy()
        if len(df) < n_before:
            print(f"[{loss_name}] filtrado: {n_before - len(df)} linhas com y<=0")

    sampler = optuna.samplers.TPESampler(seed=42, gamma=0.1, n_startup_trials=50, multivariate=True, group=True)
    study = optuna.create_study(direction=_DIRECTION[prob_score], sampler=sampler,
                                study_name=experiment_name)

    def objective(trial):
        cfg = _cap_for_vram(cfg_fn(trial))
        nf = None
        try:
            model = cls(h=h, loss=loss_params, alias=model_name,
                        **cfg, **_trainer_kwargs(model_name))
            nf = NeuralForecast(models=[model], freq=freq)
            cv = nf.cross_validation(df=df, n_windows=n_windows, val_size=val_size)

            y = cv["y"].to_numpy()
            loc = cv[_find_param_col(cv, model_name, "-loc")].to_numpy()
            scale = cv[_find_param_col(cv, model_name, "-scale")].to_numpy()

            metrics = _prob_scores_normal(y, loc, scale)     # crps + loglik
            metrics.update(_compute_metrics(y, loc))         # erros (média = loc), referência

            with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
                mlflow.log_params(trial.params)
                mlflow.set_tag("trial_number", trial.number)
                for m, v in metrics.items():
                    mlflow.log_metric(m, v)

            for m, v in metrics.items():
                trial.set_user_attr(m, v)
            return metrics[prob_score]

        except Exception as e:                # noqa: BLE001 - precisamos inspecionar OOM
            if _is_oom(e):
                print(f"[trial {trial.number}] OOM (batch={cfg.get('batch_size')}, "
                      f"input={cfg.get('input_size')}) — podando e seguindo.")
                raise optuna.TrialPruned()
            if _is_divergence(e):
                print(f"[trial {trial.number}] divergência numérica (NaN/Inf) "
                      f"(lr={cfg.get('learning_rate'):.2e}, scaler={cfg.get('scaler_type')}) "
                      f"— podando e seguindo.")
                raise optuna.TrialPruned()
            raise
        finally:
            del nf
            _free_gpu()                       # libera VRAM SEMPRE, entre trials

    with mlflow.start_run(run_name=f"{model_name}__{loss_name}__{prob_score}") as parent:
        mlflow.set_tags({"model": model_name, "loss": loss_name,
                         "distribution": distribution,
                         "optimize_metric": prob_score, "mode": "prob_score",
                         "precision": PRECISION})
        mlflow.log_params({"h": h, "num_trials": num_trials,
                           "val_size": val_size, "n_windows": n_windows, "freq": freq,
                           "precision": PRECISION,
                           "max_batch_size": MAX_BATCH_SIZE,
                           "max_input_size": MAX_INPUT_SIZE})
        study.optimize(objective, n_trials=num_trials)

        completed = [t for t in study.trials if t.value is not None]
        if not completed:
            print("[aviso] nenhum trial concluiu (todos OOM?). "
                  "Reduza MAX_BATCH_SIZE/MAX_INPUT_SIZE ou use precision menor.")
            return study, parent.info.run_id

        best = study.best_trial
        mlflow.log_params({f"best__{k}": v for k, v in best.params.items()})
        for m, v in best.user_attrs.items():
            mlflow.log_metric(f"best_{m}", float(v))
        mlflow.set_tag("best_trial_number", best.number)
        _log_optuna_plots(study)
        run_id = parent.info.run_id

    print(f"\nMelhor trial #{best.number} | {prob_score}={best.value:.6f}")
    print("Scores do melhor trial:", best.user_attrs)
    print("MLflow run_id (pai):", run_id)
    return study, run_id

## Configuração do Experimento

In [ ]:
# =============================================================================
# CONFIGURAÇÃO DO EXPERIMENTO  ->  escolha aqui ANTES de rodar
# =============================================================================
H = 15

# 3) Orçamento de busca e split de validação (defaults; podem ser sobrescritos por experimento)
NUM_TRIALS  = 5            # nº de trials do Optuna por modelo
VAL_SIZE    = H * 4        # tamanho da janela de validação usada na cross_validation
FREQ        = "D"

# 4) Onde o MLflow grava os resultados (file store './mlruns' foi descontinuado)
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"   # ou servidor remoto
BASE_EXPERIMENT_NAME = "forecast_global"

# -----------------------------------------------------------------------------
# 5) EXPERIMENTOS A ENCADEAR  ->  lista de modelos rodados EM SEQUÊNCIA
# -----------------------------------------------------------------------------
# Cada item é um dict: {"model": <chave de MODEL_REGISTRY>, "loss": <chave de LOSSES>}
# Campos opcionais por experimento: "num_trials" (sobrescreve NUM_TRIALS).
# Para rodar UM SÓ modelo, deixe a lista com um único item.
EXPERIMENTS = [
    {"model": "NHITS",   "loss": "normal"},
    {"model": "TFT",     "loss": "normal"},
    {"model": "PatchTST","loss": "normal", "num_trials": 5},
]

# Compatibilidade: MODEL_NAME/LOSS_NAME apontam para o 1º experimento (usados em
# nomes default de registro quando se roda um modelo isolado).
MODEL_NAME = EXPERIMENTS[0]["model"]
LOSS_NAME  = EXPERIMENTS[0]["loss"]
EXPERIMENT_NAME     = f"{BASE_EXPERIMENT_NAME}__{MODEL_NAME}__{LOSS_NAME}"
REGISTERED_MODEL_NAME = f"{MODEL_NAME}_{LOSS_NAME}_forecast"

print("Modelos disponíveis :", list(MODEL_REGISTRY))
print("Losses disponíveis  :", list(LOSSES))
print("-" * 60)
print("Experimentos encadeados:")
for i, exp in enumerate(EXPERIMENTS, 1):
    print(f"  {i}. modelo={exp['model']:<12} loss={exp['loss']:<8} "
          f"num_trials={exp.get('num_trials', NUM_TRIALS)}")
print(f"MLflow tracking_uri : {MLFLOW_TRACKING_URI}")

# Validação imediata das escolhas (falha cedo se algo estiver errado)
for exp in EXPERIMENTS:
    assert exp["model"] in MODEL_REGISTRY, f"model inválido: {exp['model']}"
    assert exp["loss"]  in LOSSES,         f"loss inválido: {exp['loss']}"


In [14]:
Y_df_train = pd.read_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_train_transformed_15.parquet')
Y_df_valid = pd.read_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_valid_future_15.parquet')
Y_df_test = pd.read_parquet('/home/lisan/Modelos/neural_macro_model/datasets/Y_df_test_15.parquet')

In [ ]:
# =============================================================================
# ORQUESTRADOR — encadeia vários (modelo, loss) EM SEQUÊNCIA
# Para cada experimento: sweep (Optuna) -> teste -> registro no MLflow.
# Robusto: se um modelo falhar/divergir, registra o erro e segue para o próximo.
# =============================================================================
def run_experiment_pipeline(experiments, Y_df_train, futr_df, y_true_df,
                            prob_score=PROB_SCORE, h=H,
                            num_trials=NUM_TRIALS, val_size=VAL_SIZE, freq=FREQ,
                            tracking_uri=MLFLOW_TRACKING_URI,
                            base_experiment_name=BASE_EXPERIMENT_NAME,
                            sort_by=None):
    """Roda em sequência uma lista de experimentos e devolve um leaderboard.

    `experiments`: lista de dicts {"model", "loss", "num_trials"?}.
    Retorna (leaderboard_df, resultados), onde resultados[chave] tem
    {study, nf, pred, test_metrics, model_uri, best_params}.
    """
    sort_by = sort_by or prob_score      # ordena o ranking pelo score otimizado
    resultados, linhas = {}, []

    for i, exp in enumerate(experiments, 1):
        model_name = exp["model"]
        loss_name  = exp.get("loss", "normal")
        n_trials   = exp.get("num_trials", num_trials)
        chave      = f"{model_name}__{loss_name}"
        exp_name   = f"{base_experiment_name}__{chave}"

        print("\n" + "=" * 80)
        print(f"[{i}/{len(experiments)}] EXPERIMENTO: {chave}  (num_trials={n_trials})")
        print("=" * 80)

        try:
            study, _ = run_sweep_prob(
                Y_df=Y_df_train, model_name=model_name, loss_name=loss_name,
                prob_score=prob_score, h=h, num_trials=n_trials,
                val_size=val_size, freq=freq,
                experiment_name=exp_name + f"__{prob_score}",
                tracking_uri=tracking_uri)

            if not [t for t in study.trials if t.value is not None]:
                print(f"[{chave}] nenhum trial concluiu — pulando teste/registro.")
                continue

            nf, pred, test_metrics, model_uri = finalize_best_model(
                study=study, model_name=model_name, loss_name=loss_name,
                train_df=Y_df_train, futr_df=futr_df, y_true_df=y_true_df,
                h=h, val_size=val_size, freq=freq,
                experiment_name=exp_name, tracking_uri=tracking_uri,
                registered_model_name=f"{model_name}_{loss_name}_forecast")

            resultados[chave] = {"study": study, "nf": nf, "pred": pred,
                                 "test_metrics": test_metrics, "model_uri": model_uri,
                                 "best_params": study.best_params}
            linhas.append({"model": model_name, "loss": loss_name, **test_metrics})

        except Exception as e:                       # noqa: BLE001 — não derruba a fila
            print(f"[{chave}] FALHOU: {type(e).__name__}: {e}")
        finally:
            _free_gpu()                              # libera VRAM entre modelos

    leaderboard = pd.DataFrame(linhas)
    if not leaderboard.empty and sort_by in leaderboard.columns:
        ascending = sort_by != "loglik"             # loglik: maior é melhor
        leaderboard = leaderboard.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

    print("\n" + "#" * 80)
    print(f"PIPELINE CONCLUÍDO — {len(resultados)}/{len(experiments)} modelos avaliados.")
    print("#" * 80)
    return leaderboard, resultados


## Treino, Teste e Registro (encadeado)

In [ ]:
# =============================================================================
# Avaliação no TESTE + registro do melhor modelo no MLflow Model Registry
# =============================================================================
import os
import tempfile
import mlflow.pyfunc

REGISTERED_MODEL_NAME = f"{MODEL_NAME}_{LOSS_NAME}_forecast"


def rebuild_best_model(study, model_name, loss_name, h=H):
    """Reconstrói o modelo com os MELHORES hiperparâmetros do estudo."""
    cfg_fn = MODEL_REGISTRY[model_name]
    cfg = _cap_for_vram(cfg_fn(optuna.trial.FixedTrial(study.best_params)))
    loss_params, distribution = _make_dist_loss_with_params(loss_name)
    cls = BASE_MODELS[model_name]
    model = cls(h=h, loss=loss_params, alias=model_name, **cfg, **_trainer_kwargs(model_name))
    return model, distribution


def _coverage(df, alias, level):
    lo, hi = f"{alias}-lo-{level:.1f}", f"{alias}-hi-{level:.1f}"
    if lo in df.columns and hi in df.columns:
        return float(((df["y"] >= df[lo]) & (df["y"] <= df[hi])).mean())
    return None


def evaluate_all_metrics(pred_df, y_true_df, alias, distribution):
    """Junta previsão x teste e calcula todas as métricas."""
    m = pred_df.merge(y_true_df[["unique_id", "ds", "y"]],
                      on=["unique_id", "ds"], how="inner")
    # Remove alvos sem y real (ex.: o frame de FUTURO entra no concat com y=NaN)
    # e duplicatas geradas quando valid_future e test compartilham as mesmas datas.
    m = m.dropna(subset=["y"]).drop_duplicates(["unique_id", "ds"])
    if m.empty:
        raise ValueError(
            "Merge previsão x teste vazio após remover y NaN: as datas previstas "
            "não têm alvo real em y_true_df. Confira se Y_df_test cobre o horizonte previsto.")
    y = m["y"].to_numpy()
    metrics = {}
    if distribution == "Normal":                              # probabilísticos
        loc = m[_find_param_col(m, alias, "-loc")].to_numpy()
        scale = m[_find_param_col(m, alias, "-scale")].to_numpy()
        metrics.update(_prob_scores_normal(y, loc, scale))    # crps, loglik
    if alias in m.columns:                                    # erro de ponto (média)
        metrics.update(_compute_metrics(y, m[alias].to_numpy()))
    for lvl in (50.0, 80.0):                                  # calibração
        cov = _coverage(m, alias, lvl)
        if cov is not None:
            metrics[f"coverage_{int(lvl)}"] = cov
    return metrics, m


class NeuralForecastModel(mlflow.pyfunc.PythonModel):
    """Wrapper pyfunc: carrega o NeuralForecast salvo e prevê a partir de futr_df."""
    def load_context(self, context):
        from neuralforecast import NeuralForecast
        self.nf = NeuralForecast.load(path=context.artifacts["nf_dir"])

    def predict(self, context, model_input):
        return self.nf.predict(futr_df=model_input)


def _log_pyfunc_model(**kwargs):
    try:
        return mlflow.pyfunc.log_model(name="model", **kwargs)            # MLflow >= 3
    except TypeError:
        return mlflow.pyfunc.log_model(artifact_path="model", **kwargs)   # MLflow 2.x


def _log_metrics_safe(metrics, prefix=""):
    """Loga métricas no MLflow ignorando valores não-finitos (NaN/Inf).

    Motivo: o backend SQLite do MLflow guarda NaN como is_nan=1 e, ao registrar
    o modelo na MESMA transação (autoflush), pode colidir na UNIQUE constraint
    da tabela `metrics` (key, timestamp, step, value, is_nan, run_uuid) ->
    'IntegrityError: UNIQUE constraint failed'. Além disso, métrica NaN não
    carrega informação. Então logamos só as finitas e sinalizamos as demais
    como tag, sem derrubar a run.
    """
    finite, nonfinite = {}, []
    for k, v in metrics.items():
        key = f"{prefix}{k}"
        if v is not None and np.isfinite(v):
            finite[key] = float(v)
        else:
            nonfinite.append(key)
    if finite:
        mlflow.log_metrics(finite)
    if nonfinite:
        mlflow.set_tag("metrics_nan", ",".join(nonfinite))
        print(f"[aviso] métricas não-finitas (NaN/Inf) NÃO logadas: {nonfinite}")
    return finite, nonfinite


def finalize_best_model(study, model_name, loss_name, train_df, futr_df, y_true_df,
                        h=H, val_size=None, freq="D",
                        experiment_name=None, tracking_uri=None,
                        registered_model_name=None):
    """Refit do melhor modelo -> avalia TODAS as métricas no teste -> registra no MLflow.

    Retorna (nf, pred_df, test_metrics, model_uri).
    """
    val_size = val_size if val_size is not None else h * 4
    experiment_name = experiment_name or EXPERIMENT_NAME
    registered_model_name = registered_model_name or f"{model_name}_{loss_name}_forecast"
    setup_mlflow(experiment_name, tracking_uri)

    model, distribution = rebuild_best_model(study, model_name, loss_name, h=h)
    nf = NeuralForecast(models=[model], freq=freq)

    with mlflow.start_run(run_name=f"{model_name}__{loss_name}__BEST_test") as run:
        mlflow.set_tags({"model": model_name, "loss": loss_name,
                         "distribution": distribution, "stage": "test_evaluation",
                         "best_trial_number": study.best_trial.number})
        mlflow.log_params({f"best__{k}": v for k, v in study.best_params.items()})

        try:
            nf.fit(df=train_df, val_size=val_size)
            pred = nf.predict(futr_df=futr_df)
        finally:
            _free_gpu()

        num = pred.select_dtypes("number").to_numpy()                  # diagnóstico de NaN
        n_bad = int((~np.isfinite(num)).sum()) if num.size else 0
        if n_bad:
            print(f"[aviso] previsão contém {n_bad} valores não-finitos (NaN/Inf) — "
                  f"o modelo provavelmente DIVERGIU no refit final. As métricas "
                  f"resultantes serão NaN; revise lr/scaler ou aumente o gradient_clip.")

        test_metrics, merged = evaluate_all_metrics(pred, y_true_df, model_name, distribution)
        _log_metrics_safe(test_metrics, prefix="test_")                 # ignora NaN/Inf

        with tempfile.TemporaryDirectory() as tmp:
            csv_path = os.path.join(tmp, "test_predictions.csv")    # artefato: previsões + y
            merged.to_csv(csv_path, index=False)
            mlflow.log_artifact(csv_path)

            nf_dir = os.path.join(tmp, "nf_best")                   # registra no Model Registry
            nf.save(path=nf_dir, overwrite=True, save_dataset=True)
            info = _log_pyfunc_model(python_model=NeuralForecastModel(),
                                     artifacts={"nf_dir": nf_dir},
                                     registered_model_name=registered_model_name)
        model_uri = info.model_uri
        mlflow.set_tag("registered_model", registered_model_name)

    print("\n=== Métricas no TESTE ===")
    for k, v in test_metrics.items():
        print(f"  {k:12s}: {v:.6f}")
    print(f"\nModelo registrado: {registered_model_name}")
    print(f"model_uri        : {model_uri}")
    return nf, pred, test_metrics, model_uri


In [ ]:
# =============================================================================
# EXECUÇÃO — roda TODOS os experimentos em sequência (treino + teste + registro)
# =============================================================================
leaderboard, resultados = run_experiment_pipeline(
    experiments=EXPERIMENTS,
    Y_df_train=Y_df_train,                        # treino (transformado)
    futr_df=Y_df_valid,                           # exógenas futuras p/ prever o horizonte
    y_true_df=pd.concat([Y_df_valid, Y_df_test]), # alvo REAL do teste
    prob_score=PROB_SCORE,
    h=H,
    num_trials=NUM_TRIALS,
    val_size=VAL_SIZE,
    freq=FREQ,
    tracking_uri=MLFLOW_TRACKING_URI,
    base_experiment_name=BASE_EXPERIMENT_NAME,
)

print("\n=== LEADERBOARD (ordenado pelo score otimizado) ===")
display(leaderboard)

# Seleciona o MELHOR modelo entre todos os experimentos encadeados
if leaderboard.empty:
    raise RuntimeError("Nenhum experimento produziu resultado — verifique os logs acima.")

_melhor = leaderboard.iloc[0]
CHAVE_MELHOR = f"{_melhor['model']}__{_melhor['loss']}"
nf_best   = resultados[CHAVE_MELHOR]["nf"]
pred_test = resultados[CHAVE_MELHOR]["pred"]
model_uri = resultados[CHAVE_MELHOR]["model_uri"]
print(f"\n>>> Melhor modelo: {CHAVE_MELHOR}  ({PROB_SCORE}={_melhor[PROB_SCORE]:.6f})")
print(f">>> model_uri    : {model_uri}")


## Deploy e Predição

In [ ]:
# =============================================================================
# Deploy LOCAL do melhor modelo + PREDIÇÃO (serviço em processo)
# =============================================================================
from pathlib import Path

DEPLOY_DIR = Path("./deploy/nf_best")     # diretório local que guarda o modelo servido


def salvar_para_deploy(nf, caminho=DEPLOY_DIR):
    """Persiste o NeuralForecast (pesos + dataset) num diretório local de deploy."""
    caminho = Path(caminho)
    caminho.mkdir(parents=True, exist_ok=True)
    nf.save(path=str(caminho), overwrite=True, save_dataset=True)
    print(f"✓ modelo salvo para deploy em: {caminho.resolve()}")
    return caminho


class PrevisorLocal:
    """Serviço de previsão EM PROCESSO: carrega o modelo salvo e projeta H passos.

    Uso:
        svc  = PrevisorLocal()                       # carrega ./deploy/nf_best
        fcst = svc.prever(futr_df=futr_df)           # exógenas futuras do horizonte
        fcst = svc.prever(df=hist_df, futr_df=futr)  # histórico + exógenas (controle total)
    """
    def __init__(self, caminho=DEPLOY_DIR):
        from neuralforecast import NeuralForecast
        self.nf = NeuralForecast.load(path=str(caminho))

    def prever(self, df=None, futr_df=None):
        return self.nf.predict(df=df, futr_df=futr_df)


# Salva o melhor modelo (nf_best, vindo do leaderboard) e testa o serviço em processo
salvar_para_deploy(nf_best)
svc_local  = PrevisorLocal(DEPLOY_DIR)
fcst_local = svc_local.prever(futr_df=Y_df_valid)   # previsão de H passos à frente
print("Previsão do serviço local (em processo):")
fcst_local.head()
